In [ ]:
import kagglehub
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split



im_path = os.path.join(path, '/kaggle/input/q2-ka-ai-2026/labels.csv')
df_im = pd.read_csv(im_path)

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test  = torch.tensor(X_test.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test  = torch.tensor(y_test.values, dtype=torch.float32)

In [ ]:
# 2. Create TensorDataset objects




In [ ]:
# 3. Create DataLoaders




In [ ]:
# 4. Print shape of one batch



In [ ]:
# 5. Display sample images



In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # Third linear layer: hidden layer -> hidden layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

         # 4 linear layer: hidden layer -> hidden layer
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: hidden layer -> number of classes (logits)
        self.layer5 = nn.Linear(hidden_dim, output_dim)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # First hidden layer
        a1 = self.relu(self.layer1(x))

        # Second hidden layer
        a2 = self.relu(self.layer2(a1))

        # Third hidden layer
        a3 = self.relu(self.layer3(a2))
         # Third hidden layer
        a4= self.relu(self.layer3(a3))

        # Output layer (raw scores / logits)
        output = self.layer5(a4)  # we said we'll use softmax, where is it? ¯\(ツ)/¯

        return output


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.to(device)         # shape: (batch_size, num_features)
        y_batch = y_batch.to(device)         # shape: (batch_size,)

        # Forward pass (outputs are logits)
        outputs = model(X_batch)             # shape: (batch_size, num_classes)
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
ef validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.to(device)     # shape: (batch_size, num_features)
            y_batch = y_batch.to(device)     # shape: (batch_size,)

            # Forward pass
            outputs = model(X_batch)         # shape: (batch_size, num_classes)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # Apply Softmax to get probabilities
            probabilities = F.softmax(outputs, dim=1)

            # Pick the classes with highest probabilities
            predicted = torch.argmax(probabilities, dim=1)

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# Task 4: Define device, model, loss, optimizer:
# Define criterion (loss function)
criterion = nn.MSELoss()
# Define optimizer
optimizer = AdamW(model.parameters(), learning_rate)

In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  # Train one epoch
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # Validate
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  if (epoch + 1) % 5 == 0:
    print(
      f'Epoch [{epoch+1}/{num_epochs}], '
      f'Train Loss: {train_loss:.4f}, '
      f'Val Loss: {val_loss:.4f}'
    )

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test.numpy(), predictions.flatten(), alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), predictions.min())
max_val = max(y_test.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($)', fontsize=12)
plt.ylabel('Predicted Price ($)', fontsize=12)
plt.title('Predicted vs Actual Diamond Prices', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()